# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [17]:
import warnings
from os.path import join

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display, Markdown, display_html

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity"

In [5]:
adata_query = sc.read_h5ad(join(DATA_PATH, "01ad3cd7-3929-4654-84c0-6db05bd5fd59_processed.h5ad"))
adata_ref = sc.read_h5ad(join(DATA_PATH, "b0e547f0-462b-4f81-b31b-5b0a5d96f537_processed.h5ad"))

In [6]:
adata_query.var.set_index("gene_names", inplace=True)
adata_ref.var.set_index("gene_names", inplace=True)

In [7]:
sc.pp.normalize_total(adata_query)
sc.pp.log1p(adata_query)
sc.pp.highly_variable_genes(adata_query, n_top_genes=3000, subset=True)

sc.pp.normalize_total(adata_ref)
sc.pp.log1p(adata_ref)
sc.pp.highly_variable_genes(adata_ref, n_top_genes=3000, subset=True)

/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]
/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [10]:
cluster_mapping = pd.read_parquet(join(DATA_PATH, "model-output/cluster_mapping.parquet"))
cluster_distance = pd.read_parquet(join(DATA_PATH, "model-output/cluster_distance.parquet"))

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(cluster_mapping, threshold=0.1, n_top=5)

#### Author provided cluster labels

In [13]:
for k, v in top_n_labels.items():
    display(Markdown(f"**{k}**: {v}"))

**B_Mem**: ['IGHMhi_memory_B']

**B_Mem_Prolif_Early**: ['IGHMlo_memory_B']

**B_Mem_Prolif_Late**: ['IGHMlo_memory_B']

**B_Naive**: ['naive_B']

**B_Naive_Pool3**: ['IGHMhi_memory_B', 'B', 'IGHMlo_memory_B', 'naive_B']

...

**cM_Act_1006**: ['CD14+_Monocyte', 'Monocyte', 'CD16+_Monocyte']

**cM_IFN_1006**: ['CD14+_Monocyte', 'Monocyte']

**ncM**: ['CD16+_Monocyte']

**ncM_1006**: ['CD16+_Monocyte']

**pDC**: ['pDC']

#### Ontology mapped cluster labels

In [14]:
for k, v in top_n_labels.items():
    display(Markdown(f"**{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

**B cell**: ['memory B cell']

**B cell**: ['memory B cell']

**B cell**: ['memory B cell']

**B cell**: ['naive B cell']

**B cell**: ['memory B cell', 'B cell', 'memory B cell', 'naive B cell']

...

**classical monocyte**: ['CD14-positive monocyte', 'monocyte', 'CD14-low, CD16-positive monocyte']

**classical monocyte**: ['CD14-positive monocyte', 'monocyte']

**non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

**non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

**plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell']

## Evaluate cluster similarity

In [ ]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        avg_expression = np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten()
        
        highly_expressed_genes_idxs = np.argsort(-avg_expression)[:n_genes]
        
        highly_expressed[cluster] = {
            "gene": adata.var.index[highly_expressed_genes_idxs].tolist(),
            "average_expression": avg_expression[highly_expressed_genes_idxs]
        }

    return highly_expressed


In [15]:
METHOD = "wilcoxon"
# METHOD = "t-test_overestim_var"

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(adata_query, "cell_type_author", n_genes=25, method=METHOD)
    sc.tl.rank_genes_groups(adata_ref, "cell_type_author", n_genes=25, method=METHOD)

In [ ]:
highly_expressed_query = highly_expressed_genes(adata_query, n_genes=25)
highly_expressed_ref = highly_expressed_genes(adata_ref, n_genes=25)

In [43]:
N_GENES_TO_SHOW = 10

for k, v in top_n_labels.items():
    display(Markdown(f"# **{k}:** {v}"))

    """Overlap of DE genes."""
    display(Markdown("**Overlap DE genes:**"))

    def style_overlap(v, props=""):
        return props if v in adata_query.uns["rank_genes_groups"]["names"][k] else None

    dfs = []
    dfs.append(
        pd.DataFrame({
            "gene":  adata_query.uns["rank_genes_groups"]["names"][k],
            "logfoldchange": adata_query.uns["rank_genes_groups"]["logfoldchanges"][k],
            "pval_adj": adata_query.uns["rank_genes_groups"]["pvals_adj"][k],
            # "pval": adata_query.uns["rank_genes_groups"]["pvals"][k],
        })
        .head(N_GENES_TO_SHOW)
        .style
        .set_table_attributes("style='display:inline'")
        .set_caption(f"QUERY - {k}")
    )
    for gene in v:
        dfs.append(
            pd.DataFrame({
                "gene":  adata_ref.uns["rank_genes_groups"]["names"][gene],
                "logfoldchange": adata_ref.uns["rank_genes_groups"]["logfoldchanges"][gene],
                "pval_adj": adata_ref.uns["rank_genes_groups"]["pvals_adj"][gene],
                # "pval": adata_ref.uns["rank_genes_groups"]["pvals"][gene],
            })
            .head(N_GENES_TO_SHOW)
            .style
            .map(style_overlap, props='color:green;', subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
        )

    html = dfs[0]._repr_html_()
    for df in dfs[1:]:
        html += df._repr_html_()
    display_html(html, raw=True) 
    print()

    """Overlap of highly expressed genes."""
    display(Markdown("**Overlap highly expressed genes:**"))

    def style_overlap(v, props=""):
        return props if v in highly_expressed_query[k]["gene"] else None

    dfs = []
    dfs.append(
        pd.DataFrame(highly_expressed_query[k])
        .head(N_GENES_TO_SHOW)
        .style
        .set_table_attributes("style='display:inline'")
        .set_caption(f"QUERY - {k}")
    )
    for gene in v:
        dfs.append(
            pd.DataFrame(highly_expressed_ref[gene])
            .head(N_GENES_TO_SHOW)
            .style
            .map(style_overlap, props='color:green;', subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
        )

    html = dfs[0]._repr_html_()
    for df in dfs[1:]:
        html += df._repr_html_()
    display_html(html, raw=True)
    
    print("\n")


# **B_Mem:** ['IGHMhi_memory_B']

**Overlap DE genes:**

,gene,logfoldchange,pval_adj
0,CD79A,4.353368,0.000000
1,CD74,3.314520,0.000000
2,MS4A1,4.285315,0.000000
3,CD37,1.934701,0.000000
4,HLA-DRA,3.189950,0.000000
5,HLA-DPA1,2.384425,0.000000
6,HLA-DQA1,2.850672,0.000000
7,HLA-DPB1,2.278337,0.000000
8,BANK1,3.988228,0.000000
9,HLA-DRB1,2.073251,0.000000


**Overlap highly expressed genes:**

,gene,average_expression
0,CD74,3.891718
1,EEF1A1,3.878321
2,RPS12,3.518579
3,B2M,3.423398
4,MT-CO1,3.230339
5,RPL39,3.204606
6,RPS15A,3.184745
7,MT-CO3,3.023237
8,RPS6,2.723575
9,HLA-DRA,2.659132


...

# **pDC:** ['pDC']

**Overlap DE genes:**

,gene,logfoldchange,pval_adj
0,PLD4,7.284611,0.000000
1,ITM2C,6.029186,0.000000
2,IRF7,4.755682,0.000000
3,IRF8,5.024704,0.000000
4,JCHAIN,5.311628,0.000000
5,LILRA4,10.088428,0.000000
6,ALOX5AP,3.503833,0.000000
7,UGCG,5.247618,0.000000
8,TCF4,5.447145,0.000000
9,SERPINF1,8.159710,0.000000


**Overlap highly expressed genes:**

,gene,average_expression
0,CD74,4.043548
1,B2M,3.610593
2,EEF1A1,3.600221
3,RPS12,3.264781
4,ACTB,3.095716
5,MT-CO1,3.035181
6,RPS15A,2.822146
7,FTH1,2.794597
8,RPL39,2.728448
9,HLA-DRA,2.545401
